# Experimento: reserva dimensionada pela demanda da migração

Em vez de uma porcentagem fixa nos 3 primeiros caminhos de migração, cada cooperante reserva **só no caminho principal** (o 1º candidato) **exatamente os slots de que ainda precisa para terminar a migração antes de $t_d$**:

$$
R_o(t) = \left\lceil \frac{h \,\cdot\, (D_o - M_o(t))}{(t_d - t)\,\cdot\, f_o \,\cdot\, 12{,}5} \right\rceil
$$

- $D_o - M_o(t)$: o que falta migrar (o ISP conhece seu próprio progresso);
- $t_d - t$: o tempo que resta; $h$: tempo de retenção de uma requisição; $f_o$: fator de modulação do caminho principal.

Num enlace com vários donos, as reservas somam, limitadas ao tamanho do enlace. A reserva **cai para zero** quando o dono termina e **cresce** se ele atrasa. Continua valendo só em enlaces exclusivos do grupo (σ(e) divulgado) e só para o tráfego de fundo.

No cenário 0, terminar a tempo exige de 56% a 86% do caminho principal de cada ISP. O enlace 9–12, principal dos ISPs 2 e 3, precisaria de 227 slots de 200. Então a demanda deve reservar bem mais que 40% no começo, mas só num caminho e só enquanto falta migrar.

| Variante | O que faz |
|---|---|
| `pesos` | referência sem reserva |
| `reserva_40` / `reserva_isolado_40` | melhor porcentagem fixa até agora (já rodada) |
| `demanda` | reserva pela demanda, caminho principal, proteção cooperativa |
| `demanda_isolado` | idem, proteção individual (cada ISP nos enlaces só dele) |

Faltam 64 rodadas (as de 40% são reaproveitadas).

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from experiment_worker import run_cooperation_experiment
from run_cooperation_experiment import PROTECOES, consolidar, montar_tarefas, nome_reserva, pasta_saida

%matplotlib inline

## Parâmetros

In [ ]:
BASE_INDICES = [0]
MAX_WORKERS = min(os.cpu_count() or 4, 8)
REFERENCIA = pasta_saida(None)
# (rótulo, escopo, nome em PROTECOES)
VARIANTES = [
    ("fixa 40%", "cooperativa", nome_reserva(40)),
    ("fixa 40%", "individual", nome_reserva(40, isolado=True)),
    ("demanda", "cooperativa", "demanda"),
    ("demanda", "individual", "demanda_isolado"),
]

tarefas = []
for rotulo, escopo, nome in VARIANTES:
    pasta = pasta_saida(None, nome)
    pasta.mkdir(parents=True, exist_ok=True)
    for t in montar_tarefas(BASE_INDICES, pasta, first_fit=False):
        t.update(output_dir=str(pasta), protecao=PROTECOES[nome], nome_variante=nome)
        tarefas.append(t)
for idx, t in enumerate(tarefas):
    t.update(run_index=idx, total_runs=len(tarefas))
print(f"{len(tarefas)} rodadas pendentes | {MAX_WORKERS} processos")

## Rodar

In [ ]:
inicio = datetime.now()
falhas = []
if tarefas:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(run_cooperation_experiment, t): t for t in tarefas}
        for n, future in enumerate(as_completed(futures), start=1):
            r = future.result()
            estado = "✓" if r["success"] else "✗"
            print(f"[{n}/{len(tarefas)}] {estado} {futures[future]['nome_variante']} {r['K']}  ({datetime.now() - inicio})")
            if not r["success"]:
                falhas.append(r)
print(f"Duração: {datetime.now() - inicio} | falhas: {len(falhas)}")
for _, _, nome in VARIANTES:
    consolidar(pasta_saida(None, nome))

## Carregar

In [ ]:
def carregar(pasta, **rotulos):
    r = pd.read_csv(pasta / "resumo_por_rodada.csv")
    p = pd.read_csv(pasta / "resumo_por_isp.csv")
    filtro = lambda df: df[(df["variante"] == "cooperacao") & df["base_index"].isin(BASE_INDICES)]
    return filtro(r).assign(**rotulos), filtro(p).assign(**rotulos)

partes = [carregar(REFERENCIA, reserva="pesos", escopo="—")]
partes += [carregar(pasta_saida(None, nome), reserva=rotulo, escopo=escopo) for rotulo, escopo, nome in VARIANTES]
resumo = pd.concat([r for r, _ in partes], ignore_index=True)
por_isp = pd.concat([p for _, p in partes], ignore_index=True)
I = por_isp["isp_id"].nunique()
ref = resumo[resumo["reserva"] == "pesos"].set_index(["base_index", "K"])
ORDEM = [("pesos", "—")] + [(r, e) for r, e, _ in VARIANTES]

## Ganho vs custo

Diferenças em relação a `pesos` (mesmo $K$). `reserva média` = slots reservados, em média, nos enlaces protegidos que o tráfego de fundo encontrou.

In [ ]:
linhas = []
for (reserva, escopo), g in resumo[resumo["reserva"] != "pesos"].groupby(["reserva", "escopo"]):
    g = g.set_index(["base_index", "K"])
    d_eta = g["eta_todos"] - ref["eta_todos"]
    d_bn = g["B_n"] - ref["B_n"]
    todos = g[g["k"] == I]
    linhas.append({
        "reserva": reserva, "escopo": escopo,
        "d_eta médio": d_eta.mean(), "d_B_n médio": d_bn.mean(),
        "K que melhoram": int((d_eta > 1e-9).sum()), "K que pioram": int((d_eta < -1e-9).sum()),
        "eta K=I": todos["eta_todos"].mean(), "B_n K=I": todos["B_n"].mean(), "B_m K=I": todos["B_m"].mean(),
        "reserva média (slots)": g["prot_reserva_media_slots"].mean(),
    })
t = pd.DataFrame(linhas).set_index(["reserva", "escopo"])
t["ganho por ponto de B_n"] = t["d_eta médio"] / t["d_B_n médio"]
t.round(4)

## Por tamanho de $K$

In [ ]:
for col in ["eta_todos", "B_n", "B_m"]:
    print(f"\n{col}")
    display(resumo.groupby(["reserva", "escopo", "k"])[col].mean().unstack("k").round(4))

## Ganho sobre o First-Fit disaster-aware

Cada estratégia contra a referência sem reserva (`pesos`), mesmo $K$ (pontos de η total).

In [ ]:
difs = []
for (reserva, escopo), g in resumo[resumo["reserva"] != "pesos"].groupby(["reserva", "escopo"]):
    g = g.set_index(["base_index", "K"])
    d = (g["eta_todos"] - ref["eta_todos"]).rename("d").to_frame().assign(k=g["k"], reserva=reserva, escopo=escopo)
    difs.append(d.reset_index())
ganho_vs_ff = pd.concat(difs, ignore_index=True)
ganho_vs_ff.pivot_table(index=["escopo", "reserva"], columns="k", values="d", aggfunc="mean").round(4)

## Cooperantes vs isolados (conjuntos parciais)

In [ ]:
base_isp = por_isp[por_isp["reserva"] == "pesos"].set_index(["base_index", "K", "isp_id"])["eta_i"]
linhas = []
parcial = por_isp[(por_isp["reserva"] != "pesos") & (por_isp["k"] > 0) & (por_isp["k"] < I)]
for (reserva, escopo), g in parcial.groupby(["reserva", "escopo"]):
    g = g.set_index(["base_index", "K", "isp_id"])
    d = g["eta_i"] - base_isp.reindex(g.index)
    em_k = g["em_K"].astype(bool)
    linhas.append({
        "reserva": reserva, "escopo": escopo,
        "d_eta cooperantes": d[em_k].mean(), "d_eta isolados": d[~em_k].mean(),
        "pior d_eta de um isolado": d[~em_k].min(),
        "B_n cooperantes": g.loc[em_k, "B_n_i"].mean(), "B_n isolados": g.loc[~em_k, "B_n_i"].mean(),
    })
pd.DataFrame(linhas).set_index(["reserva", "escopo"]).round(4)

In [ ]:
CORES = {"fixa 40%": "#1baf7a", "demanda": "#2a78d6"}
TEXTO, GRADE, CINZA = "#52514e", "#e5e4df", "#52514e"

def estilo(ax, titulo, ylabel):
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_xlabel("|K| (provedores cooperando)", color=TEXTO)
    ax.set_ylabel(ylabel, color=TEXTO)
    ax.set_xticks(range(I + 1))
    ax.tick_params(colors=TEXTO)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, titulo, ylabel in [
    (axes[0], "eta_todos", "Fração migrada (todos os ISPs)", "η médio"),
    (axes[1], "B_n", "Bloqueio de fundo", "B_n médio"),
]:
    m = resumo[resumo["reserva"] == "pesos"].groupby("k")[col].mean()
    ax.plot(m.index, m.values, color=CINZA, linewidth=2, marker="o", markersize=8, label="First-Fit disaster-aware (sem reserva)")
    for rotulo, escopo, _ in VARIANTES:
        m = resumo[(resumo["reserva"] == rotulo) & (resumo["escopo"] == escopo)].groupby("k")[col].mean()
        coop = escopo == "cooperativa"
        ax.plot(m.index, m.values, "-" if coop else "--", color=CORES[rotulo], linewidth=2,
                marker="o" if coop else None, markersize=8, label=f"{rotulo} ({escopo})")
    estilo(ax, titulo, ylabel)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=5, bbox_to_anchor=(0.36, -0.08))

ax = axes[2]
ax.axhline(0, color=TEXTO, linewidth=1)
for rotulo, escopo, _ in VARIANTES:
    m = ganho_vs_ff[(ganho_vs_ff["reserva"] == rotulo) & (ganho_vs_ff["escopo"] == escopo)].groupby("k")["d"].mean()
    coop = escopo == "cooperativa"
    ax.plot(m.index, m.values, "-" if coop else "--", color=CORES[rotulo], linewidth=2,
            marker="o" if coop else None, markersize=8)
estilo(ax, "Ganho sobre First-Fit disaster-aware", "Δη vs pesos (sem reserva)")
plt.tight_layout()
plt.savefig(Path("output") / "reserva_demanda.png", dpi=150, bbox_inches="tight")
plt.show()

## Leitura

- **A demanda vence a fixa?** Compare `eta K=I` e `ganho por ponto de B_n`. Se a demanda der ganho parecido com menos `B_n`, reservar só o necessário, só no caminho principal, é mais eficiente.
- **`reserva média (slots)`** mostra quanto a demanda acabou reservando na prática (de 200 slots): é a "porcentagem ideal" inferida.
- Enlaces com mais donos do que capacidade (como 9–12) ficam 100% reservados enquanto ninguém termina.